# Séance 6 — Agrégation, jointures et modèle en étoile

**Durée pratique : 3 h** &nbsp;·&nbsp; Ateliers 6.1 à 6.4

## Ce que vous saurez faire à la fin

- distinguer `agg`, `transform` et `apply` et choisir le bon ;
- contrôler la cardinalité d'une jointure et diagnostiquer une explosion de lignes ;
- passer du format large au format long et inversement ;
- livrer un modèle en étoile exploitable directement par Power BI ou Tableau.

> **Convention de nommage du module.** Le code est écrit en anglais et suit la PEP 8 :
> fonctions et variables en `snake_case`, constantes en `MAJUSCULES`. Les **noms de colonnes**
> restent en français parce qu'ils viennent de la source : renommer les colonnes d'un fichier
> d'entrée est une transformation comme une autre, elle se décide et se documente, elle ne se
> fait pas par réflexe. Vous rencontrerez cette situation partout en entreprise.

In [7]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

print('Données disponibles :')
for path in sorted(RAW_DIR.glob('*')):
    print(' ', path.name)


# Parquet conserve les types (dates, entiers, catégories) là où le CSV les perd :
# c'est le format à privilégier entre deux étapes d'un pipeline. Repli automatique
# sur le CSV si pyarrow n'est pas installé.
def save_dataset(df, name):
    try:
        path = PROCESSED_DIR / f'{name}.parquet'
        df.to_parquet(path, index=False)
    except ImportError:
        path = PROCESSED_DIR / f'{name}.csv'
        df.to_csv(path, index=False)
        print('(pyarrow absent : repli sur le CSV)')
    print('écrit :', path.name, df.shape)
    return path


def load_dataset(name):
    parquet_path = PROCESSED_DIR / f'{name}.parquet'
    csv_path = PROCESSED_DIR / f'{name}.csv'
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(f'{name} introuvable : exécutez le notebook précédent')


def dataset_exists(name):
    return ((PROCESSED_DIR / f'{name}.parquet').exists()
            or (PROCESSED_DIR / f'{name}.csv').exists())

Données disponibles :
  capteurs.csv
  clients.csv
  magasins.csv
  produits.csv
  ventes_brutes.csv
  ventes_extrait.json
  ventes_extrait.xlsx


In [8]:
sales = load_dataset('ventes_transformees')
customers = pd.read_csv(RAW_DIR / 'clients.csv')
products = pd.read_csv(RAW_DIR / 'produits.csv')
stores = pd.read_csv(RAW_DIR / 'magasins.csv')

sales['date_commande'] = pd.to_datetime(sales['date_commande'], errors='coerce')

for label, frame in [('sales', sales), ('customers', customers),
                     ('products', products), ('stores', stores)]:
    print(f'{label:<10} {frame.shape}')

sales      (17609, 18)
customers  (3060, 5)
products   (39, 5)
stores     (9, 5)


---
## Atelier 6.1 — groupby : agg, transform, apply (50 min)

### Partie guidée

In [16]:
# agg : réduit chaque groupe à une ligne
by_category = sales.groupby('categorie').agg(
    nb_commandes=('id_commande', 'count'),
    ca_total=('montant', 'sum'),
    panier_moyen=('montant', 'mean'),
    panier_median=('montant', 'median'),
    quantite_totale=('quantite', 'sum'),
).round(2).sort_values('ca_total', ascending=False)

by_category

,nb_commandes,ca_total,panier_moyen,panier_median,quantite_totale
categorie,,,,,
Informatique,5362,19601304.70,3655.60,1224.16,27435
Audio,4079,9875777.28,2421.13,1103.80,15295
Mobilier,4090,9187923.22,2246.44,783.32,18569
Accessoire,4002,8895578.51,2222.78,882.25,15618


Comparez `panier_moyen` et `panier_median`. Un écart important signale une distribution
asymétrique : quelques commandes très élevées tirent la moyenne vers le haut. Publier la
seule moyenne dans un tableau de bord donnerait une image fausse du panier typique.

In [15]:
# transform : renvoie un résultat ALIGNÉ sur les lignes d'origine
sales['ca_categorie'] = sales.groupby('categorie')['montant'].transform('sum')
sales['part_dans_categorie'] = (sales['montant'] / sales['ca_categorie'] * 100).round(4)

print(sales[['categorie', 'montant', 'ca_categorie', 'part_dans_categorie']].head())
print('\nSomme des parts par catégorie :')
print(sales.groupby('categorie')['part_dans_categorie'].sum().round(2))

      categorie  montant  ca_categorie  part_dans_categorie
0  Informatique   481.98   19601304.70               0.0025
1      Mobilier   140.96    9187923.22               0.0015
2         Audio  2073.36    9875777.28               0.0210
3  Informatique  2281.00   19601304.70               0.0116
4      Mobilier  2116.10    9187923.22               0.0230

Somme des parts par catégorie :
categorie
Accessoire      100.0
Audio           100.0
Informatique    100.0
Mobilier        100.0
Name: part_dans_categorie, dtype: float64


**La règle est simple.** `agg` réduit (une ligne par groupe), `transform` conserve
(autant de lignes qu'à l'entrée). Chaque fois que vous voulez comparer une valeur à la
statistique de son groupe, c'est `transform` qu'il vous faut, jamais un `merge` sur un
résultat de `agg`.

In [13]:
# apply : le plus souple, mais le plus lent. À réserver aux cas irréductibles.
def summarize_group(group):
    return pd.Series({
        'nb': len(group),
        'top_produit': group['id_produit'].mode().iloc[0] if len(group) else None,
        'part_annulees': (group['statut'] == 'annule').mean().round(3),
    })


sales.groupby('canal').apply(summarize_group, include_groups=False)

,nb,top_produit,part_annulees
canal,,,
boutique,525,P0004,0.166
telephone,237,P0025,0.160
web,650,P0004,0.177
BOUTIQUE,724,P0024,0.182
TELEPHONE,364,P0015,0.168
WEB,1060,P0005,0.171
boutique,4684,P0018,0.161
telephone,2352,P0036,0.168
web,7013,P0014,0.171


### Partie autonome

In [ ]:
# Q1. Construire un tableau croisé : chiffre d'affaires par catégorie et par canal,
#     avec les totaux en marge. Utilisez pivot_table avec margins=True.

sales['canal'] = sales['canal'].str.strip().str.lower()

revenue_pivot = pd.pivot_table(
    sales,
    values='montant',      
    index='categorie',     
    columns='canal',      
    aggfunc='sum',         
    margins=True,         
    margins_name='Total',
)

(revenue_pivot / 1000).round(0)


# TODO

canal,boutique,telephone,web,Total
categorie,,,,
Accessoire,2391.0,2631.0,3873.0,8896.0
Audio,3005.0,929.0,5942.0,9876.0
Informatique,7398.0,5145.0,7058.0,19601.0
Mobilier,3637.0,2764.0,2787.0,9188.0
Total,16431.0,11469.0,19661.0,47561.0


In [20]:
# Q2. Pour chaque client, calculer : nombre de commandes, CA total, panier moyen,
#     date de première et de dernière commande, nombre de catégories distinctes achetées.
#     Puis identifier les dix meilleurs clients par CA.

status = sales['statut'].str.strip().str.lower()
valid_sales = sales[(status == 'livre') & sales['quantite'].between(1, 50)]

customer_summary = valid_sales.groupby('id_client').agg(
    nb_commandes=('id_commande', 'nunique'),
    ca_total=('montant', 'sum'),
    panier_moyen=('montant', 'mean'),
    premiere_commande=('date_commande', 'min'),
    derniere_commande=('date_commande', 'max'),
    nb_categories=('categorie', 'nunique'),
)
customer_summary[['ca_total', 'panier_moyen']] = customer_summary[['ca_total', 'panier_moyen']].round(2)

print('Clients :', len(customer_summary))

# Les dix meilleurs clients par CA
top_10_customers = customer_summary.sort_values('ca_total', ascending=False).head(10)
top_10_customers



# TODO

Clients : 2977


,nb_commandes,ca_total,panier_moyen,premiere_commande,derniere_commande,nb_categories
id_client,,,,,,
C00168,9,19723.83,1972.38,2024-11-02 08:17:00,2024-11-02 08:17:00,3
C01339,9,19232.97,2137.00,2024-01-17 18:57:00,2024-12-07 18:37:00,4
C02631,7,18477.10,2639.59,NaT,NaT,4
C02974,8,18348.71,2038.75,2024-02-14 20:43:00,2024-02-14 20:43:00,4
C01233,8,18215.31,2276.91,2023-02-21 20:54:00,2024-12-30 17:10:00,3
C01106,9,17755.60,1972.84,2024-08-19 08:52:00,2024-10-04 11:23:00,4
C01267,6,17672.27,2945.38,2023-06-19 12:56:00,2023-06-19 12:56:00,2
C02803,3,17474.47,4368.62,NaT,NaT,2
C02016,8,17380.64,2172.58,2023-02-15 09:32:00,2023-04-17 16:29:00,4


In [ ]:
# Q3. Avec transform, ajouter une colonne signalant les commandes dont le montant
#     dépasse de plus de 50 % le montant médian de leur catégorie.
#     Quelle part des commandes cela représente-t-il ?


sales['montant_median_categorie'] = (sales.groupby('categorie')['montant']
                                     .transform('median'))


sales['montant_eleve'] = sales['montant'] > 1.5 * sales['montant_median_categorie']

share = sales['montant_eleve'].mean()
print(f'Commandes signalées : {sales["montant_eleve"].sum()} sur {len(sales)} ({share:.1%})')
print()
print('Part par catégorie :')
print(sales.groupby('categorie')['montant_eleve'].mean().round(3))

# TODO

Commandes signalées : 5900 sur 17609 (33.5%)

Part par catégorie :
categorie
Accessoire      0.280
Audio           0.363
Informatique    0.357
Mobilier        0.339
Name: montant_eleve, dtype: float64


---
## Atelier 6.2 — Jointures et explosion de lignes (55 min)

### Partie guidée

In [22]:
print('Lignes avant jointure :', len(sales))

with_products = sales.merge(products, on='id_produit', how='left', suffixes=('', '_prod'))
print('Après jointure produits :', len(with_products))

n_orphans = with_products['libelle'].isna().sum()
print(f'Lignes sans produit correspondant : {n_orphans}',
      f'({n_orphans / len(with_products):.2%})')

unknown_ids = sales.loc[~sales['id_produit'].isin(products['id_produit']), 'id_produit']
print('Identifiants fautifs :', unknown_ids.unique())

Lignes avant jointure : 17609
Après jointure produits : 17609
Lignes sans produit correspondant : 76 (0.43%)
Identifiants fautifs : <ArrowStringArray>
['P9999']
Length: 1, dtype: str


**Le réflexe à acquérir : compter les lignes avant et après chaque jointure.** Un `left join`
conserve le nombre de lignes *seulement si* la clé est unique à droite. Si elle ne l'est pas,
le résultat se multiplie sans avertissement.

In [23]:
# La table clients est saine : sa clé primaire est bien unique.
print('Identifiants clients dupliqués :', customers['id_client'].duplicated().sum())

# On simule donc le cas d'une table de droite dégradée — situation banale dès qu'une
# extraction est relancée deux fois ou qu'un export concatène deux périodes.
customers_with_duplicates = pd.concat([customers, customers.head(50)], ignore_index=True)
exploded = sales.merge(customers_with_duplicates, on='id_client', how='left')
print(f'{len(sales)} lignes -> {len(exploded)} après jointure sur une clé non unique')
print('Aucune erreur levée. Le problème ne se verra que bien plus loin.')

Identifiants clients dupliqués : 0
17609 lignes -> 17915 après jointure sur une clé non unique
Aucune erreur levée. Le problème ne se verra que bien plus loin.


In [24]:
# Le paramètre validate transforme le problème silencieux en erreur explicite
try:
    sales.merge(customers_with_duplicates, on='id_client', how='left', validate='many_to_one')
except Exception as error:
    print(type(error).__name__, ':', error)

MergeError : Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 id_client
   C00001
   C00002
   C00003
   C00004
   C00005 ...


Prenez l'habitude de passer `validate` sur **toutes** vos jointures. Le coût est nul et
l'erreur qu'il attrape est parmi les plus coûteuses à diagnostiquer plus tard.

### Partie autonome

In [25]:
# Q4. Assembler les quatre tables en un seul DataFrame analytique :
#     sales + products + customers + stores.
#     Contraintes : vérifier le nombre de lignes après chaque étape, utiliser validate,
#     et traiter explicitement les clés orphelines (documentez votre choix).

n_start = len(sales)
print('Lignes de départ:', n_start)

product_columns = [c for c in products.columns if c != 'id_produit']
derived_columns = ['marge_unitaire', 'taux_marge', 'ca_categorie', 'part_dans_categorie',
                   'montant_median_categorie', 'montant_eleve']
facts = sales.drop(columns=product_columns + derived_columns, errors='ignore')

#  many_to_one
analytic = facts.merge(products, on='id_produit', how='left',
                       validate='many_to_one', indicator='_match_produit')
assert len(analytic) == n_start, 'la jointure produits a changé le nombre de lignes'
print('Après jointure produits   :', len(analytic))

# Clés orphelines 
orphans = analytic['_match_produit'] == 'left_only'
print('Produits orphelins        :', orphans.sum(),
      '| identifiants :', analytic.loc[orphans, 'id_produit'].unique().tolist(),
      f'| CA concerné : {analytic.loc[orphans, "montant"].sum():,.0f} EUR')

analytic['produit_inconnu'] = orphans
text_columns = ['libelle', 'categorie', 'sous_categorie']
analytic[text_columns] = analytic[text_columns].fillna('inconnu')
analytic = analytic.drop(columns='_match_produit')

analytic = analytic.merge(customers.rename(columns={'ville': 'ville_client'}),
                          on='id_client', how='left', validate='many_to_one')
assert len(analytic) == n_start, 'la jointure clients a changé le nombre de lignes'
print('Après jointure clients    :', len(analytic),
      '| sans client :', analytic['segment'].isna().sum())

# 3. + magasins
analytic = analytic.merge(stores.rename(columns={'ville': 'ville_magasin'}),
                          on='id_magasin', how='left', validate='many_to_one')
assert len(analytic) == n_start, 'la jointure magasins a changé le nombre de lignes'
print('Après jointure magasins   :', len(analytic),
      '| sans magasin :', analytic['nom_magasin'].isna().sum())

print('\nTable analytique :', analytic.shape)
analytic.head()

# TODO

Lignes de départ: 17609
Après jointure produits   : 17609
Produits orphelins        : 76 | identifiants : ['P9999'] | CA concerné : 102,678 EUR
Après jointure clients    : 17609 | sans client : 0
Après jointure magasins   : 17609 | sans magasin : 0

Table analytique : (17609, 25)


,id_commande,date_commande,id_client,id_produit,id_magasin,quantite,prix_unitaire,remise_pct,canal,statut,ville_livraison,montant,libelle,categorie,sous_categorie,cout_achat,produit_inconnu,date_inscription,segment,ville_client,age,nom_magasin,ville_magasin,region,surface_m2
0,CMD0003112,2024-01-18 19:53:00,C02300,P0009,M05,3,160.66,5.0,telephone,retourne,Lille,481.98,Clavier modele 3,Informatique,Clavier,86.41,False,2021-04-18,Particulier,PARIS,39.0,Boutique Bordeaux,Bordeaux,Nouvelle-Aquitaine,769
1,CMD0000107,NaT,C02791,P0028,M03,2,70.48,NaN,boutique,livre,Toulouse,140.96,Lampe modele 1,Mobilier,Lampe,34.81,False,2024-02-19,Particulier,PARIS,36.0,Boutique Marseille,Marseille,PACA,460
2,CMD0010050,NaT,990031,P0017,M04,3,691.12,0.0,boutique,livre,MARSEILLE,2073.36,Enceinte modele 2,Audio,Enceinte,347.41,False,2019-02-24,Professionnel,Bordeaux,NaN,Boutique Lille,Lille,Hauts-de-France,245
3,CMD0001979,NaT,C01883,P0010,M04,4,570.25,0.0,web,retourne,Marseille,2281.00,Souris modele 1,Informatique,Souris,283.64,False,2019-06-01,Particulier,Nantes,77.0,Boutique Lille,Lille,Hauts-de-France,245
4,CMD0010513,NaT,C00571,P0024,M02,2,1058.05,10.0,web,livre,Marseille,2116.10,Bureau modele 3,Mobilier,Bureau,554.59,False,2021-01-02,Particulier,Strasbourg,44.0,Boutique Lyon,Lyon,Auvergne-Rhone-Alpes,853


In [ ]:
# Q5. Écrire une fonction safe_merge(left, right, **kwargs) qui effectue la jointure
#     et lève une exception si le nombre de lignes change alors qu'un 'left join'
#     était demandé. Elle doit aussi signaler le taux de clés non appariées.
  # TODO

def safe_merge(left, right, **kwargs): 

    how = kwargs.get('how', 'inner')             
    merged = left.merge(right, indicator='_merge_status', **kwargs)

    unmatched = (merged['_merge_status'] == 'left_only').sum()
    unmatched_rate = unmatched / len(left) if len(left) else 0.0
    print(f'safe_merge : {len(left)} -> {len(merged)} lignes | '
          f'clés non appariées : {unmatched} ({unmatched_rate:.2%})')

    # Un left join ne doit JAMAIS changer le nombre de lignes
    if how == 'left' and len(merged) != len(left):
        raise ValueError(f'left join : {len(left)} lignes attendues, {len(merged)} obtenues '
                         '(clé non unique dans la table de droite ?)')

    return merged.drop(columns='_merge_status')


# Test 1 : jointure saine avec des clés orphelines (produit P9999)
sales_without_products = sales.drop(columns=['libelle', 'categorie', 'sous_categorie',
                                             'cout_achat'], errors='ignore')
result = safe_merge(sales_without_products, products, on='id_produit', how='left')

# Test 2 : la table de droite a des doublons -> l'explosion doit être bloquée
customers_with_duplicates = pd.concat([customers, customers.head(50)], ignore_index=True)
try:
    safe_merge(sales, customers_with_duplicates, on='id_client', how='left')
except ValueError as error:
    print('Erreur levée comme prévu :', error)


safe_merge : 17609 -> 17609 lignes | clés non appariées : 76 (0.43%)
safe_merge : 17609 -> 17915 lignes | clés non appariées : 0 (0.00%)
Erreur levée comme prévu : left join : 17609 lignes attendues, 17915 obtenues (clé non unique dans la table de droite ?)


---
## Atelier 6.3 — Formats long et large (35 min)

In [28]:
# Format large : une colonne par mois, lisible par un humain
wide = sales.pivot_table(index='categorie',
                         columns=sales['date_commande'].dt.month,
                         values='montant', aggfunc='sum').round(0)
wide.columns = [f'mois_{int(month):02d}' for month in wide.columns]
wide.head()

,mois_01,mois_02,mois_03,mois_04,mois_05,mois_06,mois_07,mois_08,mois_09,mois_10,mois_11,mois_12
categorie,,,,,,,,,,,,
Accessoire,84732.0,74719.0,101603.0,72686.0,67254.0,67589.0,74614.0,41775.0,225084.0,84758.0,86707.0,110487.0
Audio,83387.0,96357.0,84796.0,826452.0,80701.0,89938.0,228162.0,74287.0,101145.0,95540.0,747281.0,128129.0
Informatique,1080410.0,791940.0,259753.0,112881.0,111009.0,153900.0,130033.0,499669.0,119101.0,145345.0,170217.0,1117781.0
Mobilier,73074.0,77671.0,66605.0,69082.0,551781.0,418702.0,74993.0,43394.0,686568.0,53219.0,89529.0,59736.0


In [29]:
# Format long : une ligne par observation, exploitable par une machine
long = wide.reset_index().melt(id_vars='categorie', var_name='mois', value_name='ca')
long.head()

,categorie,mois,ca
0,Accessoire,mois_01,84732.0
1,Audio,mois_01,83387.0
2,Informatique,mois_01,1080410.0
3,Mobilier,mois_01,73074.0
4,Accessoire,mois_02,74719.0


**Lequel choisir ?** Le format long pour tout traitement programmatique et pour les outils
de BI, qui attendent une ligne par fait. Le format large uniquement pour l'affichage final
destiné à un lecteur humain.

In [33]:
# Q6. Repasser du format long au format large sans utiliser pivot_table,
#     puis vérifier que vous retrouvez exactement le tableau de départ.

# TODO

wide_back = long.pivot(index='categorie', columns='mois', values='ca')
wide_back.columns.name = None  

print('Dimensions : long', long.shape, '-> large', wide_back.shape, '| départ', wide.shape)

pd.testing.assert_frame_equal(wide_back, wide)
print('Identique au tableau de départ :', wide_back.equals(wide))

wide_back.head(10)



Dimensions : long (48, 3) -> large (4, 12) | départ (4, 12)
Identique au tableau de départ : True


,mois_01,mois_02,mois_03,mois_04,mois_05,mois_06,mois_07,mois_08,mois_09,mois_10,mois_11,mois_12
categorie,,,,,,,,,,,,
Accessoire,84732.0,74719.0,101603.0,72686.0,67254.0,67589.0,74614.0,41775.0,225084.0,84758.0,86707.0,110487.0
Audio,83387.0,96357.0,84796.0,826452.0,80701.0,89938.0,228162.0,74287.0,101145.0,95540.0,747281.0,128129.0
Informatique,1080410.0,791940.0,259753.0,112881.0,111009.0,153900.0,130033.0,499669.0,119101.0,145345.0,170217.0,1117781.0
Mobilier,73074.0,77671.0,66605.0,69082.0,551781.0,418702.0,74993.0,43394.0,686568.0,53219.0,89529.0,59736.0


---
## Atelier 6.4 — Livrer un modèle en étoile (40 min)

Un outil de BI attend une **table de faits** (une ligne par événement mesurable, contenant
des clés étrangères et des mesures numériques) entourée de **tables de dimensions** (une ligne
par entité, contenant ses attributs descriptifs). Livrer une seule table plate oblige
l'utilisateur à refaire ce travail dans l'outil, souvent moins bien.

In [34]:
# Dimension temps : indispensable, souvent oubliée
full_range = pd.date_range(sales['date_commande'].min().normalize(),
                           sales['date_commande'].max().normalize(), freq='D')

dim_date = pd.DataFrame({'date': full_range})
dim_date['cle_date'] = dim_date['date'].dt.strftime('%Y%m%d').astype(int)
dim_date['annee'] = dim_date['date'].dt.year
dim_date['mois'] = dim_date['date'].dt.month
dim_date['nom_mois'] = dim_date['date'].dt.month_name()
dim_date['trimestre'] = dim_date['date'].dt.quarter
dim_date['semaine_iso'] = dim_date['date'].dt.isocalendar().week.astype(int)
dim_date['jour_semaine'] = dim_date['date'].dt.dayofweek
dim_date['nom_jour'] = dim_date['date'].dt.day_name()
dim_date['est_weekend'] = dim_date['jour_semaine'] >= 5

print(dim_date.shape)
dim_date.head(3)

(730, 10)


,date,cle_date,annee,mois,nom_mois,trimestre,semaine_iso,jour_semaine,nom_jour,est_weekend
0,2023-01-01,20230101,2023,1,January,1,52,6,Sunday,True
1,2023-01-02,20230102,2023,1,January,1,1,0,Monday,False
2,2023-01-03,20230103,2023,1,January,1,1,1,Tuesday,False


La dimension temps couvre **toute la plage en continu**, y compris les jours sans aucune
vente. Sans elle, un graphique d'évolution saute les jours vides et donne une impression
de continuité trompeuse.

In [36]:
# Q7. Construire le modèle complet et l'exporter en Parquet dans data/processed/star/ :
#     - fact_sales : cle_date, id_client, id_produit, id_magasin, quantite,
#       prix_unitaire, remise_pct, montant, montant_net, statut
#     - dim_date (déjà construite)
#     - dim_customer, dim_product, dim_store : dédoublonnées sur leur clé primaire
#     Vérifiez ensuite l'intégrité référentielle : toute clé de la table de faits
#     doit exister dans la dimension correspondante.

# TODO

sales = load_dataset('ventes_transformees')
sales['date_commande'] = pd.to_datetime(sales['date_commande'], format='mixed',
                                        dayfirst=True, errors='coerce')
sales = sales.drop_duplicates().dropna(subset=['date_commande'])
for column in ['canal', 'statut', 'ville_livraison']:
    sales[column] = sales[column].str.strip().str.lower()
print('Ventes :', sales.shape, '| dates manquantes :', sales['date_commande'].isna().sum())

# dim_date reconstruite sur la vraie plage de dates
full_range = pd.date_range(sales['date_commande'].min().normalize(),
                           sales['date_commande'].max().normalize(), freq='D')
dim_date = pd.DataFrame({'date': full_range})
dim_date['cle_date'] = dim_date['date'].dt.strftime('%Y%m%d').astype(int)
dim_date['annee'] = dim_date['date'].dt.year
dim_date['mois'] = dim_date['date'].dt.month
dim_date['nom_mois'] = dim_date['date'].dt.month_name()
dim_date['trimestre'] = dim_date['date'].dt.quarter
dim_date['semaine_iso'] = dim_date['date'].dt.isocalendar().week.astype(int)
dim_date['jour_semaine'] = dim_date['date'].dt.dayofweek
dim_date['nom_jour'] = dim_date['date'].dt.day_name()
dim_date['est_weekend'] = dim_date['jour_semaine'] >= 5
# ---------------------------------------------------------------------------------

fact_sales = sales.assign(
    # même format de clé que dim_date (ex. 20240118) pour pouvoir les relier
    cle_date=sales['date_commande'].dt.strftime('%Y%m%d').astype(int),
    # montant après remise (une remise absente = pas de remise)
    montant_net=sales['montant'] * (1 - sales['remise_pct'].fillna(0) / 100),
)[['cle_date', 'id_client', 'id_produit', 'id_magasin',
   'quantite', 'prix_unitaire', 'remise_pct', 'montant', 'montant_net', 'statut']]
fact_sales = fact_sales.reset_index(drop=True)

# ---------- 2. Dimensions : une ligne par entité, dédoublonnées sur la clé primaire ----------
dim_customer = customers.drop_duplicates('id_client').reset_index(drop=True)
dim_product = products.drop_duplicates('id_produit').reset_index(drop=True)
dim_store = stores.drop_duplicates('id_magasin').reset_index(drop=True)

missing_products = sorted(set(fact_sales['id_produit']) - set(dim_product['id_produit']))
print('Produits ajoutés comme membres inconnus :', missing_products)
unknown_products = pd.DataFrame({'id_produit': missing_products,
                                 'libelle': 'Produit inconnu',
                                 'categorie': 'inconnu',
                                 'sous_categorie': 'inconnu',
                                 'cout_achat': np.nan})
dim_product = pd.concat([dim_product, unknown_products], ignore_index=True)

# ---------- 3. Intégrité référentielle : chaque clé de faits existe dans sa dimension ----------
dimensions = {'cle_date': dim_date, 'id_client': dim_customer,
              'id_produit': dim_product, 'id_magasin': dim_store}

for key, dimension in dimensions.items():
    assert dimension[key].is_unique, f'{key} : clé primaire non unique dans la dimension'
    n_orphans = (~fact_sales[key].isin(dimension[key])).sum()
    print(f'{key:<11} clés orphelines : {n_orphans}')
    assert n_orphans == 0, f'{key} : {n_orphans} clés absentes de la dimension'

# ---------- 4. Export ----------
STAR_DIR = PROCESSED_DIR / 'star'
STAR_DIR.mkdir(parents=True, exist_ok=True)

tables = {'fact_sales': fact_sales, 'dim_date': dim_date, 'dim_customer': dim_customer,
          'dim_product': dim_product, 'dim_store': dim_store}
for name, table in tables.items():
    table.to_parquet(STAR_DIR / f'{name}.parquet', index=False)
    print(f'écrit : star/{name}.parquet {table.shape}')


Ventes : (17022, 18) | dates manquantes : 0
Produits ajoutés comme membres inconnus : ['P9999']
cle_date    clés orphelines : 0
id_client   clés orphelines : 0
id_produit  clés orphelines : 0
id_magasin  clés orphelines : 0
écrit : star/fact_sales.parquet (17022, 10)
écrit : star/dim_date.parquet (730, 10)
écrit : star/dim_customer.parquet (3060, 5)
écrit : star/dim_product.parquet (40, 5)
écrit : star/dim_store.parquet (9, 5)


In [38]:
# Q8. Écrire un contrôle automatique de l'intégrité référentielle :
#     pour chaque couple (table de faits, dimension), afficher le nombre de clés orphelines.
#     Le contrôle doit échouer bruyamment s'il en trouve.

  # TODO

def check_referential_integrity(fact_table, dimensions):
    """dimensions : dict {nom_colonne_cle: dataframe_dimension}"""

    report = []
    for key, dimension in dimensions.items():
        
        orphan_mask = ~fact_table[key].isin(dimension[key])
        report.append({
            'cle': key,
            'lignes_faits': len(fact_table),
            'cles_orphelines': int(orphan_mask.sum()),
            'exemples': fact_table.loc[orphan_mask, key].unique()[:5].tolist(),
           
            'cle_dimension_unique': dimension[key].is_unique,
        })

    report = pd.DataFrame(report).set_index('cle')
    print(report.to_string())

    # Échec bruyant : on lève une exception, on ne se contente pas d'un print
    problems = report[(report['cles_orphelines'] > 0) | ~report['cle_dimension_unique']]
    if not problems.empty:
        raise ValueError('Intégrité référentielle violée pour : ' + ', '.join(problems.index))

    print('\nOK — intégrité référentielle respectée')
    return report


dimensions = {'cle_date': dim_date, 'id_client': dim_customer,
              'id_produit': dim_product, 'id_magasin': dim_store}

# Test 1 : le modèle de la Q7 doit passer
check_referential_integrity(fact_sales, dimensions)

# Test 2 : on retire le membre inconnu P9999 -> le contrôle doit échouer
broken = {**dimensions, 'id_produit': dim_product[dim_product['id_produit'] != 'P9999']}
try:
    check_referential_integrity(fact_sales, broken)
except ValueError as error:
    print('\nÉchec détecté comme prévu :', error)


            lignes_faits  cles_orphelines exemples  cle_dimension_unique
cle                                                                     
cle_date           17022                0       []                  True
id_client          17022                0       []                  True
id_produit         17022                0       []                  True
id_magasin         17022                0       []                  True

OK — intégrité référentielle respectée
            lignes_faits  cles_orphelines exemples  cle_dimension_unique
cle                                                                     
cle_date           17022                0       []                  True
id_client          17022                0       []                  True
id_produit         17022               75  [P9999]                  True
id_magasin         17022                0       []                  True

Échec détecté comme prévu : Intégrité référentielle violée pour : id_produit


---
## Livrable de la séance

- `data/processed/star/` contenant la table de faits et les quatre dimensions en Parquet ;
- les fonctions `safe_merge` et `check_referential_integrity`, dans `src/` ;
- un schéma (même dessiné à la main et photographié) du modèle en étoile obtenu ;
- le compte des lignes à chaque étape de l'assemblage, avec explication des écarts.

**Ce livrable est le cœur de votre projet.** C'est le fichier que vous ouvrirez dans
Power BI ou Tableau, et celui dont la structure sera discutée en soutenance.